In [4]:
import os, sys, shutil, random, json, yaml, math, time, warnings, re
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from ultralytics import YOLO

try:
    import torch
    import ultralytics.nn.tasks as _ultra_tasks
    torch.serialization.add_safe_globals([_ultra_tasks.ClassificationModel])
except Exception:
    # If this fails, loading may still work depending on torch/ultralytics versions.
    pass
import albumentations as A
from albumentations.pytorch import ToTensorV2

try:
    from pytorch_grad_cam import EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    EigenCAM = None
    show_cam_on_image = None
    print("⚠️  pytorch-grad-cam not installed. Run: %pip install pytorch-grad-cam")

warnings.filterwarnings('ignore')

# =========================================================
# GOOGLE / COLAB
# =========================================================
# from google.colab import userdata, auth, drive
# from google import genai

# =========================================================
# HUGGING FACE
# =========================================================
# from huggingface_hub import login
# from datasets import Dataset, load_dataset

# hf_token = userdata.get('HF_TOKEN')
# login(token=hf_token)
# HF_TOKEN = hf_token

# Initialize the official Google GenAI SDK client
# GEMINI_API_KEY = userdata.get('gemini_api_key')
# gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# from google.colab import auth
# auth.authenticate_user()
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Project paths
ROOT        = Path('vim_polyp_yolo') # Changed from 'hyperkvasir_yolo'
DATA_DIR    = ROOT / 'data'
RAW_DIR     = DATA_DIR / 'raw'          # raw VIM Polyp videos & annotations
YOLO_DIR    = DATA_DIR / 'yolo'         # YOLO-formatted dataset
RESULTS     = ROOT / 'results'
MODELS_DIR  = ROOT / 'models'

for d in [RAW_DIR, YOLO_DIR/'images'/'train', YOLO_DIR/'images'/'val',
          YOLO_DIR/'images'/'test', YOLO_DIR/'labels'/'train',
          YOLO_DIR/'labels'/'val', YOLO_DIR/'labels'/'test',
          RESULTS, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Project structure created.")# ─── 5. Dataset Mappings & Verification for Classification ───────────────────
# Update project data directory to point to your clean classification folders
YOLO_DIR = (ROOT / 'yolo_classification_data').resolve()

# Verify directory exists
if YOLO_DIR.exists():
    print(f"✅ Active Dataset Path: {YOLO_DIR}\n")

    # Define dataset configuration for later dashboard tracking cells
    classify_cfg = {
        'path': str(YOLO_DIR),
        'train': 'train',
        'val': 'val',
        'test': 'test',
        'nc': NUM_CLASSES,
        'names': CLASS_NAMES,
    }

    # Save a reference classification yaml if required, though YOLO reads directories directly
    YAML_PATH = YOLO_DIR / 'dataset_classification.yaml'
    with open(YAML_PATH, 'w') as f:
        yaml.dump(classify_cfg, f, default_flow_style=False)

    print(f"✅ Classification config snapshot cached at {YAML_PATH}")
    print(yaml.dump(classify_cfg, default_flow_style=False))
else:
    print(f"⚠️  Classification folder structure not found at {YOLO_DIR}. Please run the extraction cell first.")## 2. Dataset Download — VIM Polyp

LOCAL_DATA_DIR = r"D:\Marina\colonVideosWithLabels"
# LOCAL_DATA_DIR = r"D:\Work\Nebras\colonVideosWithLabels"

# ─── 3. Segment Extraction & Clean Dataset Prep for Classification ───────────
import os
import shutil
from pathlib import Path
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit

# Ensure default constants exist
SEED = 42
if 'ROOT' not in locals():
    ROOT = Path('vim_polyp_yolo')

# 1. Configuration & Class Filters
CLASSES_TO_DROP = {'anastomosis', 'rectosigmoid', 'colon'}

all_video_files = []
unique_segments = set()

# Scan directory using os.scandir for path safety and performance
if os.path.exists(LOCAL_DATA_DIR):
    for entry in os.scandir(LOCAL_DATA_DIR):
        if entry.is_file():
            filename = entry.name

            # Skip hidden files and macOS metadata leftovers (._ files)
            if filename.startswith(".") or filename.startswith("._"):
                continue

            # Filter specifically for video formats
            if not filename.lower().endswith(('.avi', '.mp4', '.mkv')):
                continue

            stem = os.path.splitext(filename)[0]
            parts = stem.split('-')

            # Extract anatomical segment string from index 5
            if len(parts) > 5:
                segment = parts[5].lower().strip()

                if segment in CLASSES_TO_DROP:
                    continue

                unique_segments.add(segment)
                all_video_files.append((Path(entry.path), segment))

    print(f"✅ Found {len(all_video_files)} valid anatomical localization videos.")
else:
    print(f"❌ Local data directory not found at: {LOCAL_DATA_DIR}")

# 2. Dynamic Class Mapping
CLASS_NAMES = sorted(list(unique_segments))
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_ID  = {c: i for i, c in enumerate(CLASS_NAMES)}

print(f"\nClasses ({NUM_CLASSES}):")
for i, c in enumerate(CLASS_NAMES):
    print(f"   [{i}] {c}")

PALETTE = plt.colormaps['tab10']
CLASS_COLORS = {cls: PALETTE(i % 10)[:3] for i, cls in enumerate(CLASS_NAMES)}

# 3. Artifact Filtering Function
def is_unwanted_frame(frame, brightness_thresh=15, variance_thresh=5):
    """
    Returns True if the frame is pitch black, a solid blue screen,
    or a blank color artifact lacking diagnostic information.
    """
    means, stddevs = cv2.meanStdDev(frame)

    # 1. Pitch-black frames
    if max(means) < brightness_thresh:
        return True

    # 2. Solid/blank screens
    if max(stddevs) < variance_thresh:
        return True

    # 3. Explicit check for digital blue screen artifacts (High Blue, low Green/Red)
    blue_mean = means[0][0]
    green_mean = means[1][0]
    red_mean = means[2][0]
    if blue_mean > 150 and green_mean < 40 and red_mean < 40:
        return True

    return False

# 4. Setup temporary folder for frame extraction before splitting
TEMP_FRAMES_DIR = ROOT / 'data' / 'temp_frames'
TEMP_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# 5. Extract Valid Frames from Videos
all_pairs = []
skipped_artifacts = 0

if all_video_files:
    for video_path, segment in tqdm(all_video_files, desc="Extracting clean frames for localization"):
        video_id = video_path.stem

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        frame_interval = max(1, int(fps / 1))  # Extract 1 frame per second of video
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:
                # Filter out unwanted black/blue artifact screens
                if is_unwanted_frame(frame):
                    skipped_artifacts += 1
                    frame_count += 1
                    continue

                frame_name = f"{video_id}_frame_{frame_count:05d}.jpg"
                frame_path = TEMP_FRAMES_DIR / frame_name

                # Save valid frame image
                cv2.imwrite(str(frame_path), frame)
                all_pairs.append((frame_name, segment))

            frame_count += 1
        cap.release()

    print(f"\n✅ Total valid extracted frames: {len(all_pairs)}")
    print(f"🗑️ Skipped {skipped_artifacts} unwanted artifact/blank frames.")
else:
    print("⚠️ No video files to process.")

# 6. Group-Aware Video-Level Split (Prevents Data Leakage)
# ──────────────────────────────────────────────────────────────────────────
splits = {'train': [], 'val': [], 'test': []}

if all_pairs:
    frame_names = [p[0] for p in all_pairs]
    frame_labels = [p[1] for p in all_pairs]
    video_ids = [name.rsplit('_frame_', 1)[0] for name in frame_names]

    df_pairs = pd.DataFrame({
        'frame_name': frame_names,
        'label': frame_labels,
        'video_id': video_ids
    })

    # First Split: Train vs Temp (Val + Test) based on Video ID
    gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, temp_idx = next(gss_outer.split(df_pairs, groups=df_pairs['video_id']))

    train_df = df_pairs.iloc[train_idx].reset_index(drop=True)
    temp_df = df_pairs.iloc[temp_idx].reset_index(drop=True)

    # Second Split: Val vs Test based on Video ID
    gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(gss_inner.split(temp_df, groups=temp_df['video_id']))

    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

    # Construct split mapping for organizing files
    splits = {
        'train': list(zip(train_df['frame_name'], train_df['label'])),
        'val':   list(zip(val_df['frame_name'], val_df['label'])),
        'test':  list(zip(test_df['frame_name'], test_df['label'])),
    }

    print(f"\n📊 Group-Based Split Summary (by Video ID):")
    print(f"   Train videos: {train_df['video_id'].nunique():<3} | Frames: {len(train_df)}")
    print(f"   Val videos:   {val_df['video_id'].nunique():<3} | Frames: {len(val_df)}")
    print(f"   Test videos:  {test_df['video_id'].nunique():<3} | Frames: {len(test_df)}")

# 7. Organize into YOLO Classification Folder Structure
YOLO_CLASSIFY_DIR = ROOT / 'yolo_classification_data'

if all_pairs:
    for split_name, split_data in splits.items():
        for frame_name, segment_label in tqdm(split_data, desc=f"Organizing {split_name} split"):
            dest_dir = YOLO_CLASSIFY_DIR / split_name / segment_label
            dest_dir.mkdir(parents=True, exist_ok=True)

            src_path = TEMP_FRAMES_DIR / frame_name
            dest_path = dest_dir / frame_name

            if src_path.exists():
                shutil.copy2(str(src_path), str(dest_path))

    # Clean up temp folder now that everything's been copied out
    if TEMP_FRAMES_DIR.exists():
        shutil.rmtree(TEMP_FRAMES_DIR)
    print(f"\n✅ Balanced YOLO classification structure successfully built at: {YOLO_CLASSIFY_DIR.absolute()}")

🖥️  Device: cuda
   GPU: Quadro M2200
   VRAM: 4.3 GB
✅ Project structure created.
⚠️  Classification folder structure not found at D:\Marina\Nebras_RandD\vim_polyp_yolo\yolo_classification_data. Please run the extraction cell first.
✅ Found 121 valid anatomical localization videos.

Classes (8):
   [0] ascending
   [1] cecum
   [2] descending
   [3] hepaticflexure
   [4] rectum
   [5] sigmoid
   [6] splenicflexure
   [7] transverse


Extracting clean frames for localization:   0%|          | 0/121 [00:00<?, ?it/s]

Extracting clean frames for localization: 100%|██████████| 121/121 [08:12<00:00,  4.07s/it]



✅ Total valid extracted frames: 9419
🗑️ Skipped 1043 unwanted artifact/blank frames.

📊 Group-Based Split Summary (by Video ID):
   Train videos: 84  | Frames: 4868
   Val videos:   18  | Frames: 2333
   Test videos:  19  | Frames: 2218


Organizing test split: 100%|██████████| 2218/2218 [00:04<00:00, 496.71it/s]



✅ Balanced YOLO classification structure successfully built at: d:\Marina\Nebras_RandD\vim_polyp_yolo\yolo_classification_data


In [6]:
# ─── Experiment 2, Step 1: Rebuild Per-Video Frame Sequences ────────────────
# Reconstructs ordered (video_id -> frames) sequences from the split folders
# already created in cell 7. Windows are built within a single split only,
# so no video ever crosses train/val/test — same leakage protection as before.

import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image

WINDOW_SIZE = 12     # frames per sequence (~12 sec at your 1 fps extraction)
STRIDE = 6            # step between windows (50% overlap)

FRAME_IDX_RE = re.compile(r'_frame_(\d+)')

def extract_frame_idx(filename):
    m = FRAME_IDX_RE.search(filename)
    return int(m.group(1)) if m else 0

def build_video_sequences(split_dir, class_names):
    """
    Scans YOLO_DIR/<split>/<class>/*.jpg and groups them back into
    ordered per-video sequences, then slices into fixed-length windows.
    Returns a list of dicts: {video_id, class_name, frame_paths (ordered)}
    """
    video_frames = {}  # video_id -> list of (frame_idx, path, class_name)

    for class_name in class_names:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for img_path in class_dir.glob('*.jpg'):
            video_id = img_path.stem.rsplit('_frame_', 1)[0]
            frame_idx = extract_frame_idx(img_path.name)
            video_frames.setdefault(video_id, []).append((frame_idx, img_path, class_name))

    windows = []
    for video_id, frames in video_frames.items():
        frames.sort(key=lambda x: x[0])  # restore temporal order
        class_name = frames[0][2]        # single segment label per video
        paths = [f[1] for f in frames]

        for start in range(0, max(1, len(paths) - WINDOW_SIZE + 1), STRIDE):
            window_paths = paths[start:start + WINDOW_SIZE]
            if len(window_paths) < WINDOW_SIZE:
                continue  # drop short trailing windows
            windows.append({
                'video_id': video_id,
                'class_name': class_name,
                'frame_paths': window_paths,
            })

    return windows

train_windows = build_video_sequences(YOLO_DIR / 'train', CLASS_NAMES)
val_windows   = build_video_sequences(YOLO_DIR / 'val', CLASS_NAMES)
test_windows  = build_video_sequences(YOLO_DIR / 'test', CLASS_NAMES)

print(f"✅ Train windows: {len(train_windows)}")
print(f"✅ Val windows:   {len(val_windows)}")
print(f"✅ Test windows:  {len(test_windows)}")

✅ Train windows: 694
✅ Val windows:   364
✅ Test windows:  344


In [7]:
# ─── Experiment 2, Step 2: Sequence Dataset ─────────────────────────────────

IMG_SIZE = 224

seq_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SegmentSequenceDataset(Dataset):
    def __init__(self, windows, class_to_id, transform):
        self.windows = windows
        self.class_to_id = class_to_id
        self.transform = transform

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        imgs = []
        for p in w['frame_paths']:
            img = Image.open(p).convert('RGB')
            imgs.append(self.transform(img))
        seq_tensor = torch.stack(imgs, dim=0)  # (WINDOW_SIZE, C, H, W)
        label = self.class_to_id[w['class_name']]
        return seq_tensor, label

train_seq_ds = SegmentSequenceDataset(train_windows, CLASS_TO_ID, seq_transform)
val_seq_ds   = SegmentSequenceDataset(val_windows, CLASS_TO_ID, seq_transform)
test_seq_ds  = SegmentSequenceDataset(test_windows, CLASS_TO_ID, seq_transform)

BATCH_SIZE_SEQ = 4  # sequences are memory-heavy (BATCH x WINDOW frames each) — start small

train_seq_loader = DataLoader(train_seq_ds, batch_size=BATCH_SIZE_SEQ, shuffle=True, num_workers=2)
val_seq_loader   = DataLoader(val_seq_ds, batch_size=BATCH_SIZE_SEQ, shuffle=False, num_workers=2)
test_seq_loader  = DataLoader(test_seq_ds, batch_size=BATCH_SIZE_SEQ, shuffle=False, num_workers=2)

print(f"✅ Train batches: {len(train_seq_loader)} | Val: {len(val_seq_loader)} | Test: {len(test_seq_loader)}")

✅ Train batches: 174 | Val: 91 | Test: 86


In [8]:
# ─── Experiment 2, Step 3: CNN Backbone + LSTM Sequence Classifier ──────────

class CNNSequenceClassifier(nn.Module):
    def __init__(self, num_classes, hidden_size=256, lstm_layers=1,
                 freeze_backbone=True, bidirectional=True):
        super().__init__()

        # CNN backbone: ResNet18, ImageNet-pretrained, used as a per-frame
        # feature extractor. Swap for your YOLO backbone later if desired —
        # ResNet18 here keeps this cell self-contained and easy to debug first.
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
        self.feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )

        lstm_out_dim = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T_, C, H, W = x.shape
        x = x.view(B * T_, C, H, W)
        feats = self.backbone(x)                # (B*T, feature_dim)
        feats = feats.view(B, T_, self.feature_dim)  # (B, T, feature_dim)

        lstm_out, (h_n, c_n) = self.lstm(feats)
        # Use the last time step's output (both directions if bidirectional)
        seq_repr = lstm_out[:, -1, :]            # (B, lstm_out_dim)

        return self.classifier(seq_repr)

DEVICE_SEQ = DEVICE if 'DEVICE' in locals() else ('cuda' if torch.cuda.is_available() else 'cpu')

seq_model = CNNSequenceClassifier(
    num_classes=NUM_CLASSES,
    hidden_size=256,
    lstm_layers=1,
    freeze_backbone=True,   # start frozen; unfreeze later for fine-tuning if needed
    bidirectional=True,
).to(DEVICE_SEQ)

print(seq_model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Mohamed.Motear/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:48<00:00, 967kB/s] 


CNNSequenceClassifier(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=

In [ ]:
# ─── Experiment 2, Step 4: Train the Sequence Classifier ───────────────────

import torch.optim as optim
from tqdm.auto import tqdm

optimizer_seq = optim.AdamW(
    filter(lambda p: p.requires_grad, seq_model.parameters()),
    lr=1e-3, weight_decay=1e-4,
)
criterion_seq = nn.CrossEntropyLoss()
scheduler_seq = optim.lr_scheduler.CosineAnnealingLR(optimizer_seq, T_max=20)

EPOCHS_SEQ = 20

def run_epoch(model, loader, optimizer, criterion, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for seqs, labels in tqdm(loader, leave=False):
            seqs, labels = seqs.to(DEVICE_SEQ), labels.to(DEVICE_SEQ)

            if train:
                optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * seqs.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += seqs.size(0)

    return total_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(1, EPOCHS_SEQ + 1):
    train_loss, train_acc = run_epoch(seq_model, train_seq_loader, optimizer_seq, criterion_seq, train=True)
    val_loss, val_acc = run_epoch(seq_model, val_seq_loader, optimizer_seq, criterion_seq, train=False)
    scheduler_seq.step()

    print(f"Epoch {epoch:2d}/{EPOCHS_SEQ} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(seq_model.state_dict(), str(RESULTS / 'seq_model_best.pt'))
        print(f"  ✅ New best val acc {val_acc:.4f} — checkpoint saved.")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

  0%|          | 0/174 [00:00<?, ?it/s]

In [ ]:
# ─── Experiment 2, Step 5: Test-Set Evaluation ──────────────────────────────

seq_model.load_state_dict(torch.load(str(RESULTS / 'seq_model_best.pt')))
test_loss, test_acc = run_epoch(seq_model, test_seq_loader, optimizer_seq, criterion_seq, train=False)
print(f"Sequence model — Held-out test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")